In [ ]:
# 폰트 설치

!sudo apt-get -y install fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from google.colab import drive
import re
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")
import lightgbm as lgb
plt.rc('font', family='NanumBarunGothic')
import itertools

In [ ]:
drive.mount('/content/drive')

In [ ]:

# --- 1. 데이터 로드 ---
fresh_df = pd.read_csv('/content/drive/MyDrive/프로젝트/유통_수요예측/물류센터데이터 2021~2024/A물류센터 매출_2021~2024_신선식품_전처리.csv', encoding='utf-8')
temp_df = pd.read_csv('/content/drive/MyDrive/프로젝트/유통_수요예측/외생변수/기온 데이터.csv', encoding='utf-8')
precip_df = pd.read_csv('/content/drive/MyDrive/프로젝트/유통_수요예측/외생변수/강수량.csv', encoding='euc-kr')
cpi_df = pd.read_csv('/content/drive/MyDrive/프로젝트/유통_수요예측/외생변수/소비자물가지수(주요품목별)_20251009080243.csv', encoding='utf-8')

# 2. 매출 데이터 월별 집계
fresh_df['판매일'] = pd.to_datetime(fresh_df['판매일'])
sales_df = fresh_df[fresh_df['구분'] == '매출'] # '매출' 데이터만 필터링
monthly_sales = fresh_df.groupby('연월')['판매수량'].sum().reset_index()

# 3. 외부 데이터 형식 통일
# 기온 데이터
temp_df['year'] = temp_df['year'].str.strip().str.replace('"', '')
temp_df.rename(columns={'year': '연월', 'average temp': '평균기온'}, inplace=True)
temp_df = temp_df[temp_df['연월'].str.match(r'\d{4}-\d{2}')]

# 강수량 데이터
precip_df['연월'] = pd.to_datetime(precip_df['연월'], format='%b-%y').dt.strftime('%Y-%m')
precip_df.rename(columns={'강수량(mm)': '강수량'}, inplace=True)

# 소비자물가지수 데이터
cpi_df['YEAR'] = cpi_df['YEAR'].str.replace(' ', '').str.replace('.', '-')
cpi_df.rename(columns={'YEAR': '연월', 'CPI': '소비자물가지수'}, inplace=True)

# 4. 모든 데이터 병합
merged_df = pd.merge(monthly_sales, temp_df, on='연월', how='left')
merged_df = pd.merge(merged_df, precip_df, on='연월', how='left')
merged_df = pd.merge(merged_df, cpi_df, on='연월', how='left')

# 5. 최종 데이터셋 가공
merged_df['연월'] = pd.to_datetime(merged_df['연월'])
merged_df.sort_values('연월', inplace=True)
merged_df.fillna(method='ffill', inplace=True) # 결측치는 바로 이전 달의 값으로 채움
merged_df.set_index('연월', inplace=True) # '연월'을 인덱스로 설정

print("\n--- ✅ 전처리 및 통합 완료된 데이터 (상위 5개) ---")
print(merged_df.head())

In [ ]:
# -------------------------------------------------------------------
# [Cell 3] '신선식품' 데이터 탐색 및 시각화 (EDA)
# -------------------------------------------------------------------

# 1. 월별 판매수량 전체 추이 시각화
plt.figure(figsize=(16, 6))
plt.plot(merged_df.index, merged_df['판매수량'], marker='o', linestyle='-')
plt.title('A물류센터 [신선식품] 월별 판매수량 추이 (2021-2024)', fontsize=18, pad=20)
plt.xlabel('연월', fontsize=12)
plt.ylabel('총 판매수량', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 2. 월별 계절성 분석 (Box Plot)
plt.figure(figsize=(12, 7))
sns.boxplot(x=merged_df.index.month, y='판매수량', data=merged_df)
plt.title('[신선식품] 월별 판매량 계절성 분석', fontsize=18, pad=20)
plt.xlabel('월', fontsize=12)
plt.ylabel('판매수량', fontsize=12)
plt.xticks(ticks=range(12), labels=[f'{i+1}월' for i in range(12)])
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 3. 주요 변수 간 상관관계 분석 (Heatmap)
plt.figure(figsize=(10, 8))
correlation_matrix = merged_df[['판매수량', '평균기온', '강수량', '소비자물가지수']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('[신선식품] 주요 변수 간 상관관계 히트맵', fontsize=18, pad=20)
plt.show()

In [ ]:

# 1. '신선식품' 데이터 파일을 다시 불러옵니다.

fresh_df = pd.read_csv('/content/drive/MyDrive/프로젝트/유통_수요예측/물류센터데이터 2021~2024/A물류센터 매출_2021~2024_신선식품_전처리.csv', encoding='utf-8')
# '매출' 데이터만 필터링
fresh_df = fresh_df[fresh_df['구분'] == '매출'].copy()


# 2. [핵심] '상품명'을 기반으로 '상품 종류' 카테고리 생성
def categorize_product(product_name):
    if '만두' in product_name:
        return '만두류'
    if '피자' in product_name:
        return '피자류'
    if '핫도그' in product_name:
        return '핫도그류'
    if '떡갈비' in product_name or '너비아니' in product_name or '동그랑땡' in product_name:
        return '가공육류'
    if '돈까스' in product_name or '돈카츠' in product_name:
        return '돈까스류'
    if '볶음밥' in product_name:
        return '냉동밥류'
    if '두부' in product_name:
        return '두부'
    if '콩나물' in product_name or '숙주' in product_name:
        return '콩나물/숙주'
    if '김치' in product_name:
        return '김치'
    if '어묵' in product_name:
        return '어묵'
    if '면' in product_name or '우동' in product_name or '국수' in product_name or '수제비' in product_name:
        return '면/수제비류'
    return '기타' # 위에 해당하지 않는 상품은 '기타'로 분류

fresh_df['상품종류'] = fresh_df['상품명'].apply(categorize_product)


# 3. 분석 및 시각화
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# 3-1. 가장 많이 팔린 상위 15개 품목 (개별 상품 기준)
top_products = fresh_df.groupby('상품명')['판매수량'].sum().sort_values(ascending=False).head(15)
sns.barplot(x=top_products.values, y=top_products.index, ax=axes[0], palette='viridis')
axes[0].set_title('가장 많이 팔린 신선식품 TOP 15 (개별 상품명 기준)', fontsize=16)
axes[0].set_xlabel('총 판매수량', fontsize=12)
axes[0].set_ylabel('상품명', fontsize=12)

# 3-2. 가장 많이 팔린 상위 10개 카테고리 (상품 종류 기준)
top_categories = fresh_df.groupby('상품종류')['판매수량'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_categories.values, y=top_categories.index, ax=axes[1], palette='plasma')
axes[1].set_title('가장 많이 팔린 신선식품 카테고리 TOP 10', fontsize=16)
axes[1].set_xlabel('총 판매수량', fontsize=12)
axes[1].set_ylabel('상품 종류', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
try:
    plt.rc('font', family='NanumBarunGothic')
except:
    print("한글 폰트가 설치되지 않았을 수 있습니다. 이전 폰트 설치 코드를 다시 확인해주세요.")


# 1. '신선식품' 데이터 파일을 불러옵니다.
try:
    fresh_df = pd.read_csv('A물류센터 매출_2021~2024_신선식품_전처리.csv', encoding='utf-8')
    print("✅ '신선식품' 데이터를 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"🚨 파일을 읽는 중 오류가 발생했습니다: {e}")


# 2. 데이터 전처리
fresh_df['판매일'] = pd.to_datetime(fresh_df['판매일'])
fresh_df = fresh_df[fresh_df['구분'] == '매출'].copy() # 매출 데이터만 필터링

# 3. [핵심] 1월, 2월, 3월 데이터만 필터링
q1_df = fresh_df[fresh_df['판매일'].dt.month.isin([1, 2, 3])]

# 4. 1~3월 동안 가장 많이 팔린 상위 20개 품목 집계
top_20_products_q1 = q1_df.groupby('상품명')['판매수량'].sum().sort_values(ascending=False).head(20)

# 5. 시각화
plt.figure(figsize=(12, 10))
sns.barplot(x=top_20_products_q1.values, y=top_20_products_q1.index, palette='coolwarm')
plt.title('1분기(1~3월) 베스트 셀러 신선식품 TOP 20', fontsize=18, pad=20)
plt.xlabel('누적 판매수량 (2021-2024년, 1~3월)', fontsize=12)
plt.ylabel('상품명', fontsize=12)
plt.show()

print("\n" + "="*60)
print("### 1분기(1~3월) Top 20 판매 품목 리스트 ###")
print("="*60)
print(top_20_products_q1)
print("="*60)

In [ ]:

# 1. '신선식품' 데이터 파일을 불러옵니다.
try:
    # 2. 데이터 전처리 (월별 판매량 집계)
    fresh_df['판매일'] = pd.to_datetime(fresh_df['판매일'])
    sales_df = fresh_df[fresh_df['구분'] == '매출'].copy()
    monthly_sales = sales_df.groupby(pd.to_datetime(sales_df['판매일']).dt.to_period('M'))['판매수량'].sum().reset_index()
    monthly_sales.rename(columns={'판매일': '연월'}, inplace=True)

    monthly_sales['year'] = monthly_sales['연월'].dt.year
    monthly_sales['month'] = monthly_sales['연월'].dt.month

    # 3. 1월, 2월, 3월 데이터만 필터링
    q1_sales = monthly_sales[monthly_sales['month'].isin([1, 2, 3])]

    # 4. 연도별/월별 판매량 비교를 위한 피벗 테이블 생성
    q1_pivot = q1_sales.pivot_table(index='year', columns='month', values='판매수량')
    q1_pivot.columns = ['1월', '2월', '3월'] # 컬럼명 변경

    print("\n" + "="*50)
    print("### 연도별 1~3월 '신선식품' 판매수량 ###")
    print("="*50)
    print(q1_pivot.applymap('{:,.0f}'.format)) # 천 단위 쉼표 추가하여 출력
    print("="*50)

    # 5. 시각화
    q1_pivot.plot(kind='bar', figsize=(12, 7), rot=0, width=0.8)
    plt.title('연도별 1~3월 신선식품 판매수량 비교', fontsize=16)
    plt.xlabel('연도', fontsize=12)
    plt.ylabel('총 판매수량', fontsize=12)
    plt.legend(title='월')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

except FileNotFoundError:
    print("🚨 파일 경로를 찾을 수 없습니다.")
except Exception as e:
    print(f"🚨 파일을 처리하는 중 오류가 발생했습니다: {e}")

In [ ]:
# --- 2. 학습(train) 및 검증(test) 데이터 분리 ---
train = y[:'2023-12-31']  # 2021년~2023년 데이터로 모델 학습
test = y['2024-01-01':]   # 2024년 데이터로 모델 성능 검증

print(f"\n학습 데이터 기간: {train.index.min().strftime('%Y-%m')} ~ {train.index.max().strftime('%Y-%m')}")
print(f"검증 데이터 기간: {test.index.min().strftime('%Y-%m')} ~ {test.index.max().strftime('%Y-%m')}")


# --- 3. 최적의 SARIMA 파라미터 탐색 (학습 데이터 기준) ---
print("\n--- 🤖 최적의 SARIMA 파라미터를 탐색합니다. ---")
p = d = q = range(0, 2)
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], 12) for x in pdq]
best_aic = float("inf")
best_params = {}

for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            temp_model = SARIMAX(train, order=param, seasonal_order=param_seasonal)
            results = temp_model.fit(disp=False)
            if results.aic < best_aic:
                best_aic = results.aic
                best_params = {'order': param, 'seasonal_order': param_seasonal}
        except:
            continue

print(f"\n최적 파라미터: {best_params} (AIC: {best_aic:.2f})")


# --- 4. 모델 학습, 예측 및 성능 평가 ---
# 최적 파라미터로 학습 데이터에 대한 모델을 학습합니다.
model = SARIMAX(train, **best_params)
results = model.fit(disp=False)
print("\n--- ✅ 학습 데이터로 모델 학습 완료 ---")

# 검증 데이터 기간(2024년)을 예측합니다.
predictions = results.get_prediction(start=test.index.min(), end=test.index.max())
y_pred = predictions.predicted_mean

# 성능 평가 (MAE)
mae = mean_absolute_error(test, y_pred)
print("\n" + "="*50)
print(f"### 🏆 SARIMA 모델 성능 평가 ###")
print(f"2024년 예측 평균 절대 오차 (MAE): {mae:,.2f}")
print("="*50)


# --- 5. 성능 시각화 ---
plt.figure(figsize=(16, 7))
test.plot(label='Actual Sales (2024)', marker='o')
y_pred.plot(label='Predicted Sales (2024)', linestyle='--', marker='x')
plt.title('SARIMA 모델 예측 성능 (실제값 vs 예측값)', fontsize=16)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# --- 2. 시계열 교차 검증 설정 및 실행 ---
from sklearn.model_selection import TimeSeriesSplit


tscv = TimeSeriesSplit(n_splits=3, test_size=12)
mae_scores = []
predictions_all_folds = pd.Series(dtype='float64') # 모든 예측 결과를 저장할 Series
fold_number = 1

print(f"\n--- 🤖 시계열 교차 검증(K-Fold)을 시작합니다 (총 {tscv.get_n_splits()} Fold) ---")

for train_index, test_index in tscv.split(y):
    train = y.iloc[train_index]
    test = y.iloc[test_index]

    print(f"\n--- [Fold {fold_number}] ---")
    print(f"학습 기간: {train.index.min().strftime('%Y-%m')} ~ {train.index.max().strftime('%Y-%m')}")
    print(f"검증 기간: {test.index.min().strftime('%Y-%m')} ~ {test.index.max().strftime('%Y-%m')}")

    # 최적 파라미터 탐색
    p = d = q = range(0, 2)
    pdq = list(itertools.product(p, d, q))
    seasonal_pdq = [(x[0], x[1], x[2], 12) for x in pdq]
    best_aic = float("inf")
    best_params = {}
    for param in pdq:
        for param_seasonal in seasonal_pdq:
            try:
                temp_model = SARIMAX(train, order=param, seasonal_order=param_seasonal)
                results = temp_model.fit(disp=False)
                if results.aic < best_aic:
                    best_aic = results.aic; best_params = {'order': param, 'seasonal_order': param_seasonal}
            except: continue

    # 모델 학습 및 예측
    model = SARIMAX(train, **best_params)
    results = model.fit(disp=False)
    predictions = results.get_forecast(steps=len(test))
    y_pred = predictions.predicted_mean

    # 성능 평가 및 결과 저장
    mae = mean_absolute_error(test, y_pred)
    mae_scores.append(mae)
    predictions_all_folds = pd.concat([predictions_all_folds, y_pred])
    print(f"✅ Fold {fold_number} MAE: {mae:,.2f}")
    fold_number += 1

# --- 3. 최종 교차 검증 결과 출력 ---
print("\n" + "="*50)
print("### 🏆 최종 교차 검증 성능 ###")
print(f"평균 MAE: {np.mean(mae_scores):,.2f}")
print("="*50)


# --- 4. [핵심] 교차 검증 결과 종합 시각화 ---
plt.figure(figsize=(18, 8))
# 전체 실제 판매량 그리기
plt.plot(y, label='Actual Sales (전체)', color='black', linewidth=2)
# 각 Fold 별 예측 결과 그리기
predictions_all_folds.plot(label='Cross-Validated Predictions', linestyle='--', marker='x', color='red')

plt.title('SARIMA 모델 교차 검증(Cross-Validation) 결과', fontsize=18)
plt.xlabel('연월', fontsize=12)
plt.ylabel('판매수량', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
# 각 Fold의 검증 구간을 시각적으로 표시
plt.axvspan(y.iloc[tscv.split(y).__next__()[1]].index.min(), y.iloc[tscv.split(y).__next__()[1]].index.max(), color='grey', alpha=0.2, label='Fold 1 Test')
plt.axvspan(y.iloc[list(tscv.split(y))[1][1]].index.min(), y.iloc[list(tscv.split(y))[1][1]].index.max(), color='blue', alpha=0.1, label='Fold 2 Test')
plt.axvspan(y.iloc[list(tscv.split(y))[2][1]].index.min(), y.iloc[list(tscv.split(y))[2][1]].index.max(), color='green', alpha=0.1, label='Fold 3 Test')
plt.legend()
plt.show()


In [ ]:
# --- 3. 최종 모델 학습 및 2025년 예측 ---
# 최적 파라미터로 전체 데이터에 대한 최종 모델을 학습합니다.
final_model = SARIMAX(y, **best_params)
results = final_model.fit(disp=False)
print("\n--- ✅ 전체 데이터로 최종 모델 학습 완료 ---")

# get_forecast를 사용하여 2025년 1, 2, 3월을 예측합니다.
forecast = results.get_forecast(steps=3)
final_predictions = forecast.predicted_mean

# 최종 예측 결과 출력
print("\n" + "="*60)
print("### 🏆 '신선식품' 2025년 1~3월 최종 예측 (SARIMA) ###")
print("="*60)
for date, value in final_predictions.items():
    print(f"  - {date.strftime('%Y년 %m월')} 예측 판매 수량: {int(round(value)):,} 개")
print("="*60)